In [1]:
from graphdatascience import GraphDataScience
import pandas as pd, numpy as np, matplotlib.pyplot as plt
gds = GraphDataScience("bolt://localhost:7687", auth=("neo4j", "Lastfm_2026_pw"))

G, _ = gds.graph.project.native("listen_c", ["User", "Artist"],
    {"LISTEN_TO": {"orientation": "UNDIRECTED"}}, overwrite=True)
print(G.node_count(), G.relationship_count())

lab = gds.run_cypher("MATCH (n) RETURN id(n) AS nodeId, labels(n)[0] AS label, n.id AS id, n.name AS name")

19914 185668


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=1, column=18, offset=17>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 17, 'line': 1, 'column': 18}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n) RETURN id(n) AS nodeId, labels(n)[0] AS label, n.id AS id, n.name AS name'


In [2]:
runs = []
for i in range(5):
    r = gds.louvain.stream(G)                       # columns: nodeId, communityId
    r = r.merge(lab, on="nodeId")
    runs.append(r)
base = runs[0]
valid = base[~((base.label == "Artist") & (base.name.isna()) & False)]   # keep all, filter below

def summarize(df):
    # drop singleton communities made of the 390 tag-only artists
    sizes = df.groupby("communityId").size()
    big = sizes[sizes > 1]
    return sizes, big

sizes, big = summarize(base)
print("Louvain communities (all):", len(sizes), "| with >1 node:", len(big))
print(big.sort_values(ascending=False).head(10))

# modularity via stats (stored in the first call's result object)
stats = gds.louvain.stats(G)
print("Louvain modularity:", round(stats.modularity, 4), "| communities:", stats.community_count)

 Louvain:   0%|          | 0.1/100 [00:00<?, ?%/s]

 Louvain:   2%|2         | 2.45/100 [00:00<?, ?%/s]

 Louvain:   2%|2         | 2.45/100 [00:00<?, ?%/s]

 Louvain:   2%|2         | 2.0/100 [00:00<?, ?%/s]

 Louvain:   3%|2         | 2.84/100 [00:00<?, ?%/s]

Louvain communities (all): 415 | with >1 node: 25
communityId
3        3791
10       3289
2516     2533
409      2114
459      2068
899      1400
384      1035
18391     525
1631      480
1481      480
dtype: int64


 Louvain:   2%|1         | 1.55/100 [00:00<?, ?%/s]

Louvain modularity: 0.5226 | communities: 417


In [3]:
gds.label_propagation.mutate(G, mutate_property="lpa")
mod = gds.modularity.stats(G, community_property="lpa")
print("LPA modularity:", round(mod.modularity, 4), "| communities:", mod.community_count)

lpa = gds.label_propagation.stream(G).merge(lab, on="nodeId")
lpa_sizes = lpa.groupby("communityId").size()
print("LPA communities with >1 node:", int((lpa_sizes > 1).sum()))
print(lpa_sizes.sort_values(ascending=False).head(10))

 Label Propagation:  18%|#8        | 18.18/100 [00:00<?, ?%/s]

LPA modularity: 0.3636 | communities: 502


 Label Propagation:  36%|###6      | 36.36/100 [00:00<?, ?%/s]

LPA communities with >1 node: 112
communityId
1906    12164
1896     4103
1908      102
1601       57
1995       51
1674       50
1930       48
1552       48
6963       47
1668       47
dtype: int64


In [6]:
from sklearn.metrics import normalized_mutual_info_score as nmi, adjusted_rand_score as ari

def aligned(a, b):
    m = a[["nodeId", "communityId"]].merge(b[["nodeId", "communityId"]], on="nodeId", suffixes=("_a", "_b"))
    return m.communityId_a, m.communityId_b

print("Louvain vs Louvain (run 1 vs runs 2-5):")
for i in range(1, 5):
    a, b = aligned(runs[0], runs[i]); print(" run", i+1, "NMI", round(nmi(a, b), 3), "ARI", round(ari(a, b), 3))

a, b = aligned(runs[0], lpa)
print("Louvain vs LPA: NMI", round(nmi(a, b), 3), "ARI", round(ari(a, b), 3))

Louvain vs Louvain (run 1 vs runs 2-5):
 run 2 NMI 0.662 ARI 0.607
 run 3 NMI 0.662 ARI 0.607
 run 4 NMI 0.662 ARI 0.607
 run 5 NMI 0.663 ARI 0.614
Louvain vs LPA: NMI 0.462 ARI 0.188


In [7]:
part = runs[0][["nodeId", "id", "label", "communityId"]].copy()
sizes0 = part.groupby("communityId").size()
order = sizes0[sizes0 > 1].sort_values(ascending=False).index.tolist()
cname = {c: f"C{i+1}" for i, c in enumerate(order)}          # C1 = biggest
part["comm"] = part.communityId.map(cname).fillna("single")
part.to_csv("data/louvain_run1.csv", index=False)

for lbl in ["User", "Artist"]:
    rows = part[part.label == lbl][["id", "comm"]].to_dict("records")
    for i in range(0, len(rows), 5000):
        gds.run_cypher(f"UNWIND $rows AS r MATCH (n:{lbl} {{id: r.id}}) SET n.louvain = r.comm",
                       {"rows": rows[i:i+5000]})
print(part.comm.value_counts().head(8))

comm
C1    3791
C2    3289
C3    2533
C4    2114
C5    2068
C6    1400
C7    1035
C8     525
Name: count, dtype: int64


In [8]:
m = part.merge(lpa[["nodeId", "communityId"]].rename(columns={"communityId": "lpa"}), on="nodeId")
top_lpa = lpa.groupby("communityId").size().nlargest(2).index.tolist()
for c in list(cname)[:7]:
    sub = m[m.communityId == c]
    print(cname[c], len(sub), "nodes | in LPA giant:", round((sub.lpa == top_lpa[0]).mean()*100, 1),
          "% | in LPA #2:", round((sub.lpa == top_lpa[1]).mean()*100, 1), "%")

C1 3791 nodes | in LPA giant: 96.0 % | in LPA #2: 2.2 %
C2 3289 nodes | in LPA giant: 88.2 % | in LPA #2: 10.3 %
C3 2533 nodes | in LPA giant: 1.2 % | in LPA #2: 98.7 %
C4 2114 nodes | in LPA giant: 71.0 % | in LPA #2: 14.0 %
C5 2068 nodes | in LPA giant: 86.8 % | in LPA #2: 0.8 %
C6 1400 nodes | in LPA giant: 51.6 % | in LPA #2: 0.4 %
C7 1035 nodes | in LPA giant: 44.2 % | in LPA #2: 29.1 %


In [9]:
tags = pd.read_csv("data/tags.dat", sep="\t", encoding="latin-1")
tg = gds.run_cypher("MATCH (u:User)-[t:TAGGED]->(a:Artist) RETURN a.id AS id, t.tagID AS tagID")
tg = tg.merge(part[part.label == "Artist"][["id", "comm"]], on="id").merge(tags, on="tagID")

lis = gds.run_cypher("MATCH (:User)-[:LISTEN_TO]->(a:Artist) RETURN a.id AS id, count(*) AS listeners")
art = (part[part.label == "Artist"].merge(lis, on="id")
          .merge(lab[lab.label == "Artist"][["id", "name"]], on="id"))

for c in list(cname.values())[:7]:
    a = art[art.comm == c].sort_values("listeners", ascending=False)
    nu = int(((part.label == "User") & (part.comm == c)).sum())
    print(f"\n=== {c}: {nu} users, {len(a)} artists")
    print("top artists:", ", ".join(a.name.head(10)))
    print("top tags:   ", ", ".join(tg[tg.comm == c].tagValue.value_counts().head(8).index))


=== C1: 480 users, 3311 artists
top artists: The Beatles, Muse, Radiohead, Coldplay, The Killers, Depeche Mode, Pink Floyd, Nirvana, Arctic Monkeys, Placebo
top tags:    indie, rock, alternative, electronic, indie rock, alternative rock, british, female vocalists

=== C2: 359 users, 2930 artists
top artists: Linkin Park, Green Day, 30 Seconds to Mars, System of a Down, My Chemical Romance, blink-182, Rammstein, Panic! At the Disco, Marilyn Manson, Fall Out Boy
top tags:    rock, alternative, metal, alternative rock, metalcore, hardcore, indie, punk

=== C3: 485 users, 2048 artists
top artists: Lady Gaga, Britney Spears, Rihanna, Katy Perry, Madonna, Avril Lavigne, Christina Aguilera, Paramore, Beyoncé, Ke$ha
top tags:    pop, female vocalists, dance, rnb, rock, electronic, hip-hop, soul

=== C4: 145 users, 1969 artists
top artists: Duran Duran, a-ha, Pet Shop Boys, ABBA, Tears for Fears, Erasure, INXS, Enya, Hans Zimmer, Kraftwerk
top tags:    80s, new wave, electronic, pop, synthpop,

In [10]:
e = gds.run_cypher("MATCH (u:User)-[:LISTEN_TO]->(a:Artist) RETURN u.id AS u, a.id AS a")
cu = part[part.label == "User"].set_index("id").comm
ca = part[part.label == "Artist"].set_index("id").comm
e["cu"] = e.u.map(cu); e["ca"] = e.a.map(ca)

top7 = [f"C{i}" for i in range(1, 8)]
M = pd.crosstab(e.cu, e.ca)
share = (M.div(M.sum(axis=1), axis=0) * 100).round(1)
print("Rows: community of the listener. Columns: community of the artist. Values: % of listens")
print(share.loc[top7, top7])

Rows: community of the listener. Columns: community of the artist. Values: % of listens
ca    C1    C2    C3    C4    C5    C6    C7
cu                                          
C1  74.4   8.0   6.6   3.8   4.2   1.1   0.5
C2  14.3  64.1  11.6   1.6   6.2   0.3   0.5
C3   6.3   7.1  82.4   1.5   0.9   0.1   0.6
C4  19.7   6.6   8.5  56.7   5.1   1.1   0.2
C5  15.6  11.7   2.3   3.1  64.7   0.6   0.4
C6  23.4   5.5   1.3   3.6   2.7  59.6   1.3
C7  14.6   6.4  14.2   1.7   2.8   1.6  55.8


In [11]:
a = e.assign(external=(e.cu != e.ca)).groupby("a").agg(
        listeners=("u", "size"), ext_share=("external", "mean")).reset_index()
a = a.merge(part[part.label == "Artist"][["id", "comm"]], left_on="a", right_on="id")
a = a.merge(lab[lab.label == "Artist"][["id", "name"]], on="id")

def mix(aid):
    s = e[e.a == aid].cu.value_counts(normalize=True).head(4)
    return ", ".join(f"{k} {v*100:.0f}%" for k, v in s.items())

cand = a[a.listeners >= 50].sort_values("ext_share", ascending=False).head(12)
cand["listener_mix"] = cand.a.apply(mix)
print(cand[["name", "comm", "listeners", "ext_share", "listener_mix"]].to_string(index=False))

print("\nFor comparison, the most LOCAL artists with >= 50 listeners:")
loc = a[a.listeners >= 50].sort_values("ext_share").head(8)
print(loc[["name", "comm", "listeners", "ext_share"]].to_string(index=False))

         name comm  listeners  ext_share                   listener_mix
  James Blunt   C2         61   0.770492  C3 33%, C1 30%, C2 23%, C4 3%
      Ramones   C5        104   0.730769  C1 37%, C5 27%, C2 26%, C3 3%
         ABBA   C4         73   0.726027  C3 37%, C4 27%, C1 12%, C2 7%
        Queen   C5        208   0.721154 C5 28%, C1 28%, C4 14%, C2 13%
     Bon Jovi   C5         84   0.690476 C5 31%, C2 24%, C3 23%, C1 13%
 Jack Johnson   C2         73   0.684932  C1 33%, C2 32%, C3 11%, C4 5%
   John Mayer   C2         79   0.683544  C2 32%, C3 28%, C1 27%, C4 5%
      Misfits   C2         50   0.680000  C2 32%, C5 28%, C1 22%, C4 6%
    Metallica   C5        224   0.678571  C2 35%, C5 32%, C1 20%, C4 5%
Elvis Presley   C1         95   0.673684 C1 33%, C5 20%, C2 16%, C3 15%
Guns N' Roses   C5        161   0.664596 C5 34%, C2 30%, C1 17%, C3 11%
   Bob Marley   C1         50   0.660000 C1 34%, C7 14%, C5 14%, C2 12%

For comparison, the most LOCAL artists with >= 50 listeners:
  

In [12]:
bw = gds.betweenness.stream(G, sampling_size=2000, sampling_seed=42).merge(lab, on="nodeId")
tb = bw[bw.label == "Artist"].sort_values("score", ascending=False).head(10).merge(
        part[["nodeId", "comm"]], on="nodeId")
print(tb[["name", "comm", "score"]].to_string(index=False))

 Betweenness Centrality:   1%|1         | 1.0/100 [00:00<?, ?%/s]

          name comm        score
   The Beatles   C1 1.061108e+06
     Lady Gaga   C3 8.998555e+05
     Radiohead   C1 7.101433e+05
          Muse   C1 5.178997e+05
  Depeche Mode   C1 4.745469e+05
       Madonna   C3 4.416453e+05
      Coldplay   C1 4.362099e+05
Britney Spears   C3 3.904271e+05
      Paramore   C3 3.459755e+05
    Pink Floyd   C1 3.088056e+05


In [13]:
import os
os.makedirs("results", exist_ok=True); os.makedirs("figures", exist_ok=True)
def keep(name, fn):
    try: fn(); print("saved", name)
    except Exception as ex: print("SKIPPED", name, "->", type(ex).__name__, ex)

keep("louvain run1",   lambda: part.to_csv("results/c_louvain_run1.csv", index=False))
keep("louvain 5 runs", lambda: pd.concat([r[["nodeId", "communityId"]].assign(run=i+1) for i, r in enumerate(runs)]).to_csv("results/c_louvain_all_runs.csv", index=False))
keep("lpa",            lambda: lpa[["nodeId", "label", "id", "communityId"]].to_csv("results/c_lpa.csv", index=False))
keep("matrix",         lambda: share.loc[top7, top7].to_csv("results/c_community_matrix.csv"))
keep("bridge artists", lambda: cand.to_csv("results/c_bridge_artists.csv", index=False))
keep("local artists",  lambda: loc.to_csv("results/c_local_artists.csv", index=False))
keep("betweenness+community", lambda: tb.to_csv("results/c_top_betweenness_with_community.csv", index=False))

def fig_matrix():
    S = share.loc[top7, top7]
    plt.figure(figsize=(6, 5)); plt.imshow(S.values, cmap="Blues"); plt.colorbar(label="% of listens")
    plt.xticks(range(7), top7); plt.yticks(range(7), top7)
    for i in range(7):
        for j in range(7): plt.text(j, i, S.values[i, j], ha="center", va="center", fontsize=8)
    plt.xlabel("artist community"); plt.ylabel("listener community"); plt.title("Where each community listens")
    plt.tight_layout(); plt.savefig("figures/c_community_matrix.png", dpi=200); plt.close()
keep("figure matrix", fig_matrix)

saved louvain run1
saved louvain 5 runs
saved lpa
saved matrix
saved bridge artists
saved local artists
saved betweenness+community
saved figure matrix
